# Application Development

As per the spec, this side of the code consists of two main components: a MQTT service layer that abstracts all low level communication from the user; a lock state manager that queries and manages the connected locks.

This has been designed to be extensible, maintainable and plug-in-able.

## Service Layer

To begin, we should inspect the MQTT service layer to see it's core functionality. The service layer is dumb to lock functionality and purely provides core MQTT functionality.

It provides 6 public interfaces:
- `start` : Starts the MQTT loop and connects to the broker. After connecting the `_on_connect` callback is triggered and the relevant topics are subscribed to if that information was provided prior to connection.
- `stop`: Disconnects from the broker and stops the MQTT loop.
- `publish`: Publishes a message to provided topic. Expects the message to be a JSON serializable `dict`.
- `subscribe`: Subscribes to a topic. If client is not started or connected, subscriptions are deferred until after `on_connect` is triggered.
- `has_message`: Thread safe check of the internal message queue of received messages.
- `get_message`: Gets most oldest message in the queue. This is a FIFO so messages are appended to the end of the queue and are removed from the front. 

It also has two public properties decorated with a `@property` attribute. All of these are thread safe:
- `connected`: True if service layer is connected to broker.
- `last_message_received_time`: Last received message UNIX time stamp.

The service layer can be constructed from parameters or using the `@classmethod` decorator it can be constructed using `from_env` which will pull the parameters from environment variables. All the environment variables used are passed through the docker compose file, in production this would be done through confidential config but for demo purposes it is publicly visible.

Once again, the password must be registered in the broker server using the same docker command that was used to configure device and broker passwords.

```bash

docker run --rm -v "$PWD/broker:/mosquitto/config" eclipse-mosquitto \
  mosquitto_passwd -b -c /mosquitto/config/passwd lockmgr mstpwd

```

Received messages are parsed into a dictionary alongside its topic and other metadata.

```python
# Store alongside topic it was received on and raw message
self._message_queue.put(
    {
        "topic": msg.topic,
        "payload": payload_dict,
        "raw": msg.payload,
        "qos": msg.qos,
        "retain": msg.retain,
    }
)
```

This captures all the necessary low level interactions for the demo, any additional functionality could be added to the class but there is not much else to add without allowing lock specific scope creep in. 

## Lock Manager

The lock manager allows users to queries lock statuses, lock and unlock devices as well as maintain background updated list of device states. All connections are monitored and unresponsive devices and are marked in the state list.

It is constructed by providing a valid and constructed MQTT service layer, a list of lock ids that it is to manage and an optional poll interval for device health checks.

The lock manager has 8 public interfaces:
- `start` : Starts the MQTT service layer, main message processing loop and device polling loop.
- `stop`: Unblocks managed locks, ends the main message loop, polling loop and stops the MQTT service. 
- `register_callback` : Registers external call backs to a specified lock id to be called when a lock status changes. The callback must take a `dict` as its single parameter - this `dict` is the new device state. This intended for GUI threads or other external interactions.
- `get_status` : Gets the last received status for a lock of the given lock id. This does not query the device via the broker instead checks the last received status which was likely received in the polling loop. This is thread safe.
- `get_mqtt_metrics` : Thread safe check of the MQTT service layer metrics, includes service connection status and last received message time.
- `lock` : Sends lock command to a given lock id. Is blocking and has five second timeout. Returns the device state.
- `unlock` : Same as `lock`, but unlocks instead.
- `query_status` : Directly queries the status of a given lock id via the broker. Has a five second timeout, blocks and returns the status.

It also has a public property, `status` which returns all device status as a dict.

There are two threads (aside from the threads happening in the service layer). 

The main loop runs the `_process_messages` method which loops and polls the service layer for received messages,

```python
# Loop while running
while self._running:
    # Check for message in service layer
    if self._mqtt_service.has_message():
```

Received messages are validated simply,

```python
# Payload must be dict
if not isinstance(payload, dict):
    return None
# Should have string keys
if not all(isinstance(k, str) for k in payload.keys()):
    return None
return payload
```

If the received message contains an error message, the relevant device's state is updated accordingly in `_handle_error`,

```python
# Log error and change device state
logger.warning(f"[{lock_id}] Error from device: {error}")
with self._state_lock:
    self._device_states[lock_id]["error"] = error
    self._device_states[lock_id]["connected"] = True
    self._device_states[lock_id]["last_updated"] = time.time()

# Notify condition of response
cond = self._locks.get(lock_id)
if cond:
    with cond:
        cond.notify_all()
```

If the message is a valid device status, the state is updated and the callback (if exists) is called,

```python
# Configure the state
new_state = {
    "state": payload.get("state"),
    "battery_percent": payload.get("battery_percent"),
    "firmware_version": payload.get("firmware_version"),
    "last_updated": time.time(),
    "connected": True,
    "error": None,
}

# Update device
with self._state_lock:
    self._device_states[lock_id] = new_state

...

# Trigger callback
with self._callbacks_lock:
    callbacks = list(self._callbacks.get(lock_id, []))
for cb in callbacks:
    # Safe handler
    def safe_cb(cb=cb, state=dict(new_state)):
        try:
            cb(state)
        except Exception as e:
            logger.warning(f"Callback error for {lock_id}: {e}")

    # Call in new thread
    threading.Thread(target=safe_cb, daemon=True).start()
```

The polling thread queries the status of each device the manager is configured for, it also marks devices stale in `_check_stale_devices` if the a message was not received in an appropriate time.

```python 
while self._running:
    # Query status
    for lock_id in self._lock_ids:
        if not self._running:
            break
        self.query_status(lock_id)
    # Check for stale devices
    self._check_stale_devices(max_age=10.0)
    time.sleep(self._poll_interval)

...

# Check initial time for stale check
now = time.time()
with self._state_lock:
    # Check last updated for all devices
    for lock_id, state in self._device_states.items():
        last = state.get("last_updated")
        if last is None:
            continue
        age = now - last
        if age > max_age and state.get("connected", True):
            # Set connected state
            logger.warning(
                f"[{lock_id}] Device stale — no update in {age:.1f}s"
            )
            state["connected"] = False
            state["error"] = "stale"
```

## Docker Setup

This part is follows largely the same process as the device logic so I won't repeat myself. We simply create a docker file that copies the relevant code over and points it to a main entry point. A CLI was written in `main.py` for this purpose that allows the user to access the public interfaces via user input in a terminal. 

The `start_sim` script was updated to include a step that brings up WSL terminals through powershell (this would be simpler if I were on ubuntu) and attaching the running dockers to them. The main one of interest is the CLI docker container as this is what the user interacts with but the others are useful for debug and seeing communications flying across the broker. Change the environment variable `LOG_LEVEL` in the relevant container in the docker compose to adjust the depth of debug that is printed.

```bash
powershell.exe -Command '
  wt.exe new-tab --title "MQTT Broker" wsl -e bash -c "docker logs -f mqtt-broker" ; `
  wt.exe new-tab --title "lock-01" wsl -e bash -c "docker logs -f lock-01" ; `
  wt.exe new-tab --title "lock-02" wsl -e bash -c "docker logs -f lock-02" ; `
  wt.exe new-tab --title "lock-03" wsl -e bash -c "docker logs -f lock-03" ; `
  wt.exe new-tab --title "lock-04" wsl -e bash -c "docker logs -f lock-04" ; `
  wt.exe new-tab --title "Smart Lock CLI" wsl -e bash -c "docker attach cli"
'
```